# Notebook 03 — Regresi: Forecasting Revenue Cabang

**Fase 2 · Minilab EduBI · Data Mining**

---

## Tujuan
Membangun model regresi untuk **memperkirakan revenue cabang** pada periode berikutnya
berdasarkan data historis penjualan.

Hasil forecast membantu manajemen dalam:
- Penetapan target penjualan yang realistis
- Alokasi stok dan sumber daya per cabang
- Deteksi dini underperformance

## Alur
```
ClickHouse gold.gold_sales_daily
    ↓
Agregasi bulanan per cabang
    ↓
Feature Engineering (lag features, rolling mean)
    ↓
Linear Regression + Ridge Regression
    ↓
Evaluasi: MAE, RMSE, R²
    ↓
Visualisasi prediksi vs aktual
    ↓
Log ke MLflow
```

## Referensi
- Hoerl, A.E. & Kennard, R.W. (1970). *Ridge Regression*. Technometrics, 12(1), 55–67.
- Scikit-learn LinearModels: https://scikit-learn.org/stable/modules/linear_model.html

---
## 1. Setup & Koneksi

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import clickhouse_connect
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CH_HOST    = os.getenv('CH_HOST', 'localhost')
CH_PORT    = int(os.getenv('CH_PORT', 8123))
CH_USER    = os.getenv('CH_USER', 'default')
CH_PASS    = os.getenv('CH_PASSWORD', '')
MLFLOW_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('03_regression_revenue_forecast')

client = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASS
)
print('Koneksi ClickHouse berhasil.')

---
## 2. Load & Agregasi Data Bulanan

In [ ]:
query = """
SELECT
    branch,
    order_year   AS year,
    order_month  AS month,
    sum(total_revenue)  AS monthly_revenue,
    sum(total_orders)   AS monthly_orders
FROM gold.gold_sales_daily
GROUP BY branch, year, month
ORDER BY branch, year, month
"""

df = client.query_df(query)
df['period'] = pd.to_datetime(
    df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2) + '-01'
)
df['monthly_revenue'] = df['monthly_revenue'].astype(float)
df['monthly_orders']  = df['monthly_orders'].astype(int)

print(f'Total records: {len(df)}')
print('Cabang:', df['branch'].unique())
df.head(10)

---
## 3. Eksplorasi Tren Revenue per Cabang

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for branch, grp in df.groupby('branch'):
    ax.plot(grp['period'], grp['monthly_revenue'], marker='o', label=branch)
ax.set(title='Tren Revenue Bulanan per Cabang',
       xlabel='Periode', ylabel='Revenue (Rp)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp {x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('experiments/revenue_trend.png', dpi=100)
plt.show()

---
## 4. Feature Engineering (Lag Features)

In [ ]:
def build_lag_features(df_branch: pd.DataFrame) -> pd.DataFrame:
    """Buat fitur lag dan rolling mean untuk satu cabang."""
    d = df_branch.sort_values('period').copy()
    d['lag_1']       = d['monthly_revenue'].shift(1)   # revenue bulan lalu
    d['lag_2']       = d['monthly_revenue'].shift(2)   # revenue 2 bulan lalu
    d['rolling_3m']  = d['monthly_revenue'].shift(1).rolling(3).mean()  # rata 3 bulan
    d['month_num']   = d['period'].dt.month            # pola musiman
    return d.dropna()

df_features = pd.concat([
    build_lag_features(grp) for _, grp in df.groupby('branch')
]).reset_index(drop=True)

# Encode branch
le = LabelEncoder()
df_features['branch_enc'] = le.fit_transform(df_features['branch'])

print(f'Records setelah feature engineering: {len(df_features)}')
df_features[['branch', 'period', 'lag_1', 'lag_2', 'rolling_3m', 'monthly_revenue']].head(8)

---
## 5. Training & Evaluasi (Linear vs Ridge)

In [ ]:
FEATURES = ['lag_1', 'lag_2', 'rolling_3m', 'month_num', 'branch_enc', 'monthly_orders']
TARGET   = 'monthly_revenue'

X = df_features[FEATURES].values
y = df_features[TARGET].values

# Gunakan TimeSeriesSplit untuk validasi time-series yang benar
tscv  = TimeSeriesSplit(n_splits=3)
split_idx = list(tscv.split(X))[-1]   # ambil split terakhir
train_idx, test_idx = split_idx

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
def eval_and_log(model, name, alpha=None):
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae  = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)

        if alpha: mlflow.log_param('alpha', alpha)
        mlflow.log_metric('MAE',  mae)
        mlflow.log_metric('RMSE', rmse)
        mlflow.log_metric('R2',   r2)
        mlflow.sklearn.log_model(model, f'{name}_model')

        print(f'{name:<20} MAE: Rp {mae:,.0f}  RMSE: Rp {rmse:,.0f}  R²: {r2:.4f}')
        return y_pred

y_pred_lr  = eval_and_log(LinearRegression(), 'linear_regression')
y_pred_r01 = eval_and_log(Ridge(alpha=0.1),   'ridge_alpha_0.1', alpha=0.1)
y_pred_r1  = eval_and_log(Ridge(alpha=1.0),   'ridge_alpha_1.0', alpha=1.0)
y_pred_r10 = eval_and_log(Ridge(alpha=10.0),  'ridge_alpha_10',  alpha=10.0)

---
## 6. Visualisasi Prediksi vs Aktual

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
periods_test = df_features['period'].values[test_idx]

ax.plot(periods_test, y_test,       'ko-', label='Aktual',    linewidth=2)
ax.plot(periods_test, y_pred_lr,    'b--', label='Linear Reg', alpha=0.8)
ax.plot(periods_test, y_pred_r1,    'r--', label='Ridge α=1',  alpha=0.8)

ax.set(title='Prediksi Revenue vs Aktual',
       xlabel='Periode', ylabel='Revenue (Rp)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp {x/1e6:.1f}M'))
ax.legend()
plt.tight_layout()
plt.savefig('experiments/regression_forecast.png', dpi=100)
plt.show()

---
## 7. Kesimpulan

**Pertanyaan Diskusi:**
1. Mengapa TimeSeriesSplit lebih tepat daripada random split untuk data time-series?
2. Apa perbedaan antara Linear Regression dan Ridge Regression? Kapan Ridge lebih baik?
3. Fitur lag apa lagi yang mungkin meningkatkan akurasi prediksi?
4. Bagaimana cara mendeteksi dan menangani outlier pada data revenue?

**Lihat hasil eksperimen di MLflow:** http://localhost:5000